### Robinhood Deep Financial Analysis
This notebook uses the supervisor multi-agent system to perform a deep financial analysis of Robinhood (HOOD):
1. Create a supervisor assistant with specialized finance, research, and writing agents
2. Run a comprehensive financial analysis
3. Update the assistant for a different output format
4. Revert to the original configuration

#### Setup

In [1]:
from langgraph_sdk import get_client
from dotenv import load_dotenv
import os

# ---- COLAB SETUP ----
# If running in Google Colab, set the tunnel URL here before load_dotenv().
# Make sure cloudflared tunnel is running locally: cloudflared tunnel run dev-tunnel
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in str(globals().get("__builtins__", ""))
if IN_COLAB:
    os.environ["DEPLOYMENT_URL"] = "https://dev.crossstory.com"
    os.environ["API_KEY"] = ""
    print("✅ Colab mode — using: https://dev.crossstory.com")

# ---- SETUP ----
load_dotenv()
DEPLOYMENT_URL = os.getenv("DEPLOYMENT_URL")
API_KEY = os.getenv("API_KEY")
GRAPH_ID = "supervisor_prebuilt"

print(f"🔗 Deployment URL: {DEPLOYMENT_URL}")


🔗 Deployment URL: https://dev.crossstory.com


#### Connect and Create a Financial Analysis Assistant

In [2]:
# 1. Connect to the LangGraph server
client = get_client(url=DEPLOYMENT_URL, api_key=API_KEY)
print("🔗 Connected to LangGraph server")

# 2. Create a financial analysis assistant using the supervisor graph
print("🤖 Creating Robinhood financial analysis assistant...")

assistant = await client.assistants.create(
    graph_id=GRAPH_ID,
    config={
        "configurable": {
            "supervisor_system_prompt": """You are an expert financial analysis director orchestrating a team of specialized agents 
to produce a deep financial analysis of Robinhood Markets (HOOD).

Your workflow:
1. Route to finance_research_agent to get Robinhood's latest financial data and news (ticker: HOOD)
2. Route to general_research_agent to research Robinhood's business model, competition, and market position
3. Route to writing_agent to compile a comprehensive financial analysis report

Be thorough and strategic. Gather both quantitative (financial metrics) and qualitative (market position, risks) data.""",
            "supervisor_model": "openai/gpt-4.1",

            "finance_system_prompt": """You are an expert finance research analyst. 
Research Robinhood Markets (ticker: HOOD) thoroughly:
- Latest earnings and revenue figures
- Key financial metrics (P/E ratio, revenue growth, user growth)
- Recent news and analyst ratings
- Stock price performance
Use the finance_research tool with ticker 'HOOD' and basic_research_tool for additional context.""",
            "finance_model": "openai/gpt-4.1",
            "finance_tools": ["finance_research", "basic_research_tool", "get_todays_date"],

            "research_system_prompt": """You are an expert business and market research analyst.
Research Robinhood's competitive landscape and business fundamentals:
- Business model and revenue streams (PFOF, Gold subscriptions, crypto)
- Key competitors (Schwab, Fidelity, eToro, Webull)
- Regulatory risks and challenges
- Growth opportunities and strategic initiatives
- User base trends and demographics
Use the advanced_research_tool for comprehensive research.""",
            "research_model": "openai/gpt-4.1",
            "research_tools": ["advanced_research_tool", "get_todays_date"],

            "writing_system_prompt": """You are an expert financial analyst and report writer.
Compile all research into a structured deep financial analysis report with these sections:

# Robinhood Markets (HOOD) — Deep Financial Analysis

## 1. Company Overview
## 2. Financial Performance
   - Revenue & profitability trends
   - Key metrics and ratios
## 3. Business Model Analysis
   - Revenue streams breakdown
   - Unit economics
## 4. Competitive Position
   - Market share and differentiation
   - Key competitors
## 5. Risk Factors
   - Regulatory, market, and operational risks
## 6. Growth Outlook
   - Opportunities and catalysts
## 7. Investment Summary
   - Bull case / Bear case
   - Key takeaways

Be data-driven, specific, and professional.""",
            "writing_model": "openai/gpt-4.1",
            "writing_tools": ["get_todays_date"],
        }
    },
    name="Robinhood Financial Analyst"
)

print("✅ Assistant created successfully!")
print(f"   📍 Assistant ID: {assistant['assistant_id']}")
print(f"   📝 Name: {assistant['name']}")
print(f"   🔢 Version: {assistant['version']}")

🔗 Connected to LangGraph server
🤖 Creating Robinhood financial analysis assistant...
✅ Assistant created successfully!
   📍 Assistant ID: b1728504-a1e3-429f-adda-eba5cfe96533
   📝 Name: Robinhood Financial Analyst
   🔢 Version: 1


In [3]:
import json

def print_event(event_data, seen_ids):
    for node_name, node_data in event_data.items():
        if not isinstance(node_data, dict):
            continue
        for msg in node_data.get("messages", []):
            msg_id = msg.get("id", "")
            if msg_id and msg_id in seen_ids:
                continue
            if msg_id:
                seen_ids.add(msg_id)

            msg_type = msg.get("type")
            msg_name = msg.get("name") or node_name
            msg_content = msg.get("content", "")
            # tool_calls can be top-level or in additional_kwargs
            tool_calls = msg.get("tool_calls") or msg.get("additional_kwargs", {}).get("tool_calls")

            if msg_type == "ai":
                if tool_calls:
                    for tc in (tool_calls if isinstance(tool_calls, list) else []):
                        tc_name = tc.get("name") if isinstance(tc, dict) else tc.function.name
                        print(f"🔧 [{msg_name}] → {tc_name}")
                elif msg_content and str(msg_content).strip():
                    print(f"\n💬 [{msg_name}]:\n{msg_content}\n")

            elif msg_type == "tool":
                tool_name = msg.get("name", "tool")
                content = msg.get("content", "")
                if tool_name.startswith("transfer"):
                    continue
                try:
                    results = json.loads(content)
                    print(f"✅ Tool '{tool_name}' returned {len(results) if isinstance(results, list) else 1} results")
                except:
                    print(f"✅ Tool '{tool_name}' completed")

# 3. Run the deep financial analysis
thread = await client.threads.create()
print(f"🧵 Thread: {thread['thread_id']}")
print("📊 Running deep financial analysis of Robinhood (HOOD)...")
print("="*60)

seen_ids = set()
async for event in client.runs.stream(
    thread["thread_id"],
    assistant["assistant_id"],
    input={"messages": [{"role": "human", "content":
        "Perform a deep financial analysis of Robinhood Markets (HOOD). "
        "Research their financials, business model, competitive position, risks, and growth outlook. "
        "Compile everything into a comprehensive investment analysis report."}]},
    stream_mode="updates",
):
    if event.event == "metadata":
        print(f"📋 Run ID: {event.data.get('run_id', '')[:8]}...\n")
    elif event.event == "updates":
        print_event(event.data, seen_ids)

print("\n" + "="*60)
print("🎉 Full analysis complete!")
print("="*60)


🧵 Thread: 019df6af-f78c-7dc0-81c8-e935d9891051
📊 Running deep financial analysis of Robinhood (HOOD)...


📋 Run ID: 019df6af...



🔧 [supervisor] → transfer_to_finance_research_agent



💬 [finance_research_agent]:
Here is a comprehensive investment analysis of Robinhood Markets (NASDAQ: HOOD), focusing on their financials, business model, competitive position, risks, and growth outlook:

1. Financials (as of Q1 2025)
- Customers & Assets: Robinhood has 25.8 million funded customers (an 8% YoY increase) and manages $255 billion in assets.
- Revenue & Profit: Net income increased 114% Year-over-Year (YoY) to $336 million. Subscription services (Robinhood Gold) saw rapid growth, with 3.2 million subscribers in Q1 2025 (90% YoY growth).
- The business is showing strong top-line and bottom-line growth, with robust new customer acquisition and deepening customer relationships through premium offerings.

2. Business Model
- Robinhood pioneered commission-free, mobile-first brokerage services targeting younger and tech-savvy investors. The platform enables self-directed trading of stocks, options, ETFs, and cryptocurrencies.
- Revenue Sources: Major sources include payments 

🔧 [supervisor] → transfer_to_general_research_agent



💬 [general_research_agent]:
Here is a comprehensive investment analysis report for Robinhood Markets (NASDAQ: HOOD) based on the latest available financials, competitive landscape, business fundamentals, and growth outlook:

---

## Executive Summary

Robinhood Markets has rapidly matured from a disruptive fintech upstart into a major player in retail investing. Its user-friendly, commission-free platform has fueled significant user base growth and revenue expansion. With record profitability and strategic expansion into products and geographies, Robinhood stands at a compelling—though not risk-free—inflection point for investors.

---

## 1. Financial Performance (2023–2025)

**Revenues & Profitability:**
- 2024 revenue: $2.95 billion (+58% YoY), 2025 revenue: $4.5 billion (estimated, +~52% YoY) ([Source](https://www.businessofapps.com/data/robinhood-statistics/), [SEC filings](https://www.sec.gov/Archives/edgar/data/1783879/000178387925000045/q42024robinhoodexhibit991.htm)).
- 2024 

🔧 [supervisor] → transfer_to_writing_agent



💬 [writing_agent]:
# Robinhood Markets (HOOD) — Deep Financial Analysis

---

## 1. Company Overview

Robinhood Markets, Inc. is a U.S.-based fintech company providing commission-free trading of stocks, ETFs, options, and cryptocurrencies via an app-first, mobile-centric platform. Since its founding in 2013, Robinhood has become synonymous with democratizing investing, attracting millions of younger, tech-savvy retail customers. The company is publicly listed on NASDAQ under the ticker HOOD.

---

## 2. Financial Performance

### Revenue & Profitability Trends

- **Revenue:** 2024 revenue reached $2.95 billion (+58% YoY). 2025 revenue is estimated at $4.5 billion (+~52% YoY).
- **Profitability:** Achieved first full-year profit in 2024: net income $1.41 billion (up from -$541 million in 2023). Q1 2025 net income: $336 million (+114% YoY).
- **User Base & Assets:** 25.8 million funded customers (+8% YoY), $255 billion in assets under custody by Q1 2025.
- **Robinhood Gold:** Premium su


💬 [supervisor]:
Here is a comprehensive investment analysis report on Robinhood Markets (HOOD):

---

**Robinhood Markets (HOOD) — Deep Financial Analysis**

---

### 1. Executive Summary

Robinhood Markets continues its transformation from a disruptive fintech upstart to a major public brokerage. The company now features record profitability, robust user growth, and rapid product innovation. While its growth and expanding international reach provide clear upside, Robinhood also faces real regulatory and market risks that investors must weigh carefully.

---

### 2. Financial Performance

- **Revenue and Earnings**: 2024 revenue was $2.95 billion (up 58% YoY), with 2025 estimates approaching $4.5 billion. Net income hit $1.41 billion in 2024 (first-ever annual profit), swinging from a $541 million loss in 2023.
- **Customer Base**: 25.8 million funded customers (+8% YoY in Q1 2025), $255 billion assets under custody (AUC).
- **Profitability**: Q1 2025 net income was $336 million, up 1

In [4]:
# 4. Update the assistant to produce a concise executive summary
print("🔄 Updating assistant for executive summary format...")

updated_assistant = await client.assistants.update(
    assistant["assistant_id"],
    config={
        "configurable": {
            "supervisor_system_prompt": """You are a financial analysis director. 
Produce a concise executive summary of Robinhood (HOOD) — max 500 words.
Route to finance_research_agent first, then directly to writing_agent.""",
            "supervisor_model": "openai/gpt-4.1",
            "writing_system_prompt": """You are a financial analyst. Write a concise executive summary of Robinhood (HOOD) in max 500 words covering:
- Current financial snapshot
- Key strengths and risks  
- Investment verdict (Buy/Hold/Sell with brief rationale)""",
            "writing_model": "openai/gpt-4.1",
            "writing_tools": ["get_todays_date"],
            "finance_tools": ["finance_research", "get_todays_date"],
            "research_tools": ["advanced_research_tool", "get_todays_date"],
        }
    },
)

print("✅ Assistant updated to executive summary mode!")
print(f"   🔢 New Version: {updated_assistant['version']}")


🔄 Updating assistant for executive summary format...
✅ Assistant updated to executive summary mode!
   🔢 New Version: 2


In [5]:
# 5. Run the executive summary version
thread2 = await client.threads.create()
print(f"🧵 Thread: {thread2['thread_id']}")
print("📝 Running executive summary analysis...")
print("="*60)

seen_ids2 = set()
async for event in client.runs.stream(
    thread2["thread_id"],
    updated_assistant["assistant_id"],
    input={"messages": [{"role": "user", "content":
        "Give me a concise executive summary and investment verdict on Robinhood (HOOD)."}]},
    stream_mode="updates"
):
    if event.event == "updates":
        print_event(event.data, seen_ids2)

print("\n" + "="*60)
print("🎉 Executive summary complete!")
print("="*60)


🧵 Thread: 019df6b0-e9ab-7c73-9af3-81bcf5b26eb0
📝 Running executive summary analysis...


🔧 [supervisor] → transfer_to_finance_research_agent



💬 [finance_research_agent]:
As of today's date, May 5, 2026, there is no current financial news or recent updates available for Robinhood (HOOD) from major financial sources. This absence of news might reflect a lack of recent significant events, or it could indicate issues with data availability. 

Executive Summary:
Robinhood Markets (HOOD) operates a popular commission-free trading platform targeting retail investors. Historically, HOOD has been known for its user-friendly interface, growth in funded accounts, and influence on democratizing investing among younger demographics. However, its business model faces ongoing regulatory scrutiny, competitive pressures from traditional and emerging fintech firms, and market risks associated with its core clientele’s trading behaviors.

Investment Verdict:
Due to the lack of new, relevant financial news as of today, investors should exercise caution. Existing concerns around business sustainability, competitive environment, and regulatory r

🔧 [supervisor] → transfer_to_writing_agent



💬 [writing_agent]:
**Robinhood (HOOD) Executive Summary**

**Current Financial Snapshot**  
Robinhood Markets, Inc. (NASDAQ: HOOD) is a leading commission-free trading platform targeting retail investors, particularly millennials and Gen Z. As of its latest reported quarter (Q1 2024), Robinhood posted net revenues of approximately $618 million, up year-over-year, driven by strong interest income and growing assets under custody. The platform reported over 23 million funded accounts and roughly $115 billion in assets under custody. Historically volatile, HOOD’s profitability remains constrained with modest net income margins and continued reinvestment into growth and innovation.

**Key Strengths**  
- **Strong Brand & User Growth:** Robinhood’s reputation for zero-commission trading and an intuitive app interface continues to attract new, younger investors, sustaining a robust user base.
- **Diversified Offerings:** Expansion into options, cryptocurrencies, and retirement products enha


💬 [supervisor]:
Robinhood (HOOD) Executive Summary & Investment Verdict

Robinhood Markets, Inc. (NASDAQ: HOOD) is a pioneering fintech platform best known for commission-free trading and accessibility to retail investors, especially younger demographics. As of early 2024, Robinhood reported over 23 million funded accounts and around $115 billion in assets under custody, supported by revenue growth driven chiefly by interest income and diversified financial products (including options, crypto, and retirement accounts).

Despite its continued brand strength and user growth, Robinhood faces several headwinds. Regulatory scrutiny—especially concerning payment for order flow—poses ongoing risks to its business model. The intense competition from established financial institutions and new fintech entrants further compresses margins. Profitability remains inconsistent, with net income hampered by high operating and compliance costs, and the business remains sensitive to trading activity tha

In [6]:
# 6. Revert to the full deep analysis version
print("⏪ Reverting to full deep analysis (Version 1)...")
await client.assistants.set_latest(assistant['assistant_id'], 1)
print("✅ Reverted successfully!")
print("   🔢 Now using: Version 1 (Full deep financial analysis)")


⏪ Reverting to full deep analysis (Version 1)...
✅ Reverted successfully!
   🔢 Now using: Version 1 (Full deep financial analysis)
